# Numerical Methods in Mathematics

This notebook demonstrates various numerical methods for solving mathematical problems using Python.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import optimize, integrate
import seaborn as sns

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

## Newton's Method for Root Finding

Newton's method is an iterative algorithm for finding roots of a function:

$$x_{n+1} = x_n - \frac{f(x_n)}{f'(x_n)}$$

In [ ]:
def newton_method(f, df, x0, tol=1e-6, max_iter=100):
    """
    Newton's method for finding roots.
    
    Parameters:
    f: function to find root of
    df: derivative of f
    x0: initial guess
    tol: tolerance for convergence
    max_iter: maximum number of iterations
    """
    x = x0
    iterations = []
    
    for i in range(max_iter):
        fx = f(x)
        dfx = df(x)
        
        if abs(dfx) < 1e-12:
            raise ValueError("Derivative too small")
        
        x_new = x - fx / dfx
        iterations.append((i, x, fx, x_new))
        
        if abs(x_new - x) < tol:
            return x_new, iterations
        
        x = x_new
    
    raise ValueError("Did not converge")

# Example: Find root of f(x) = x^3 - 2x - 5
def f(x):
    return x**3 - 2*x - 5

def df(x):
    return 3*x**2 - 2

# Find root starting from x0 = 2
root, iterations = newton_method(f, df, 2.0)
print(f"Root found: {root:.6f}")
print(f"Verification: f({root:.6f}) = {f(root):.2e}")

In [ ]:
# Visualize the convergence
x_vals = [it[1] for it in iterations]
f_vals = [abs(it[2]) for it in iterations]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Plot function and iterations
x = np.linspace(1.5, 2.5, 1000)
ax1.plot(x, f(x), 'b-', label='$f(x) = x^3 - 2x - 5$', linewidth=2)
ax1.axhline(y=0, color='k', linestyle='--', alpha=0.5)
ax1.plot(x_vals, [f(x) for x in x_vals], 'ro-', markersize=8, label='Newton iterations')
ax1.plot(root, f(root), 'g*', markersize=15, label=f'Root: {root:.4f}')
ax1.set_xlabel('x')
ax1.set_ylabel('f(x)')
ax1.set_title('Newton\'s Method Convergence')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot convergence rate
ax2.semilogy(range(len(f_vals)), f_vals, 'bo-', markersize=6)
ax2.set_xlabel('Iteration')
ax2.set_ylabel('|f(x)|')
ax2.set_title('Convergence Rate')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Numerical Integration

We'll demonstrate the trapezoidal rule and Simpson's rule for numerical integration:

**Trapezoidal Rule:**
$$\int_a^b f(x) dx \approx \frac{h}{2}[f(x_0) + 2f(x_1) + 2f(x_2) + \ldots + 2f(x_{n-1}) + f(x_n)]$$

**Simpson's Rule:**
$$\int_a^b f(x) dx \approx \frac{h}{3}[f(x_0) + 4f(x_1) + 2f(x_2) + 4f(x_3) + \ldots + f(x_n)]$$

In [ ]:
def trapezoidal_rule(f, a, b, n):
    """Trapezoidal rule for numerical integration."""
    h = (b - a) / n
    x = np.linspace(a, b, n + 1)
    y = f(x)
    return h * (0.5 * y[0] + np.sum(y[1:-1]) + 0.5 * y[-1])

def simpsons_rule(f, a, b, n):
    """Simpson's rule for numerical integration (n must be even)."""
    if n % 2 != 0:
        raise ValueError("n must be even for Simpson's rule")
    
    h = (b - a) / n
    x = np.linspace(a, b, n + 1)
    y = f(x)
    
    return h/3 * (y[0] + 4*np.sum(y[1::2]) + 2*np.sum(y[2:-1:2]) + y[-1])

# Example: Integrate sin(x) from 0 to π
def test_function(x):
    return np.sin(x)

a, b = 0, np.pi
exact_value = 2.0  # Exact integral of sin(x) from 0 to π

# Test different numbers of intervals
n_values = [4, 8, 16, 32, 64, 128]
trap_errors = []
simp_errors = []

print("Integration of sin(x) from 0 to π")
print(f"Exact value: {exact_value}")
print("\nn\tTrapezoidal\tError\t\tSimpson's\tError")
print("-" * 60)

for n in n_values:
    trap_result = trapezoidal_rule(test_function, a, b, n)
    simp_result = simpsons_rule(test_function, a, b, n)
    
    trap_error = abs(trap_result - exact_value)
    simp_error = abs(simp_result - exact_value)
    
    trap_errors.append(trap_error)
    simp_errors.append(simp_error)
    
    print(f"{n}\t{trap_result:.6f}\t{trap_error:.2e}\t{simp_result:.6f}\t{simp_error:.2e}")

In [ ]:
# Visualize the error convergence
plt.figure(figsize=(10, 6))
plt.loglog(n_values, trap_errors, 'bo-', label='Trapezoidal Rule', markersize=8)
plt.loglog(n_values, simp_errors, 'rs-', label="Simpson's Rule", markersize=8)

# Add theoretical convergence rates
h_values = [(np.pi - 0) / n for n in n_values]
plt.loglog(n_values, [h**2 for h in h_values], 'b--', alpha=0.5, label='$O(h^2)$')
plt.loglog(n_values, [h**4 for h in h_values], 'r--', alpha=0.5, label='$O(h^4)$')

plt.xlabel('Number of intervals (n)')
plt.ylabel('Absolute Error')
plt.title('Convergence of Numerical Integration Methods')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Monte Carlo Integration

Monte Carlo integration uses random sampling to estimate integrals:

$$\int_a^b f(x) dx \approx \frac{b-a}{n} \sum_{i=1}^n f(x_i)$$

where $x_i$ are random points in $[a,b]$.

In [ ]:
def monte_carlo_integration(f, a, b, n):
    """Monte Carlo integration."""
    x_random = np.random.uniform(a, b, n)
    y_values = f(x_random)
    return (b - a) * np.mean(y_values)

# Estimate π using Monte Carlo integration of √(1-x²) from -1 to 1
def semicircle(x):
    return 2 * np.sqrt(1 - x**2)

# Run multiple trials
n_samples = [100, 1000, 10000, 100000, 1000000]
pi_estimates = []
errors = []

print("Estimating π using Monte Carlo integration")
print("Function: 2√(1-x²) integrated from -1 to 1")
print("\nSamples\t\tEstimate\tError")
print("-" * 35)

np.random.seed(42)  # For reproducibility

for n in n_samples:
    estimate = monte_carlo_integration(semicircle, -1, 1, n)
    error = abs(estimate - np.pi)
    
    pi_estimates.append(estimate)
    errors.append(error)
    
    print(f"{n:,}\t\t{estimate:.6f}\t{error:.6f}")

print(f"\nActual value of π: {np.pi:.6f}")

In [ ]:
# Visualize Monte Carlo convergence
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Plot estimates vs sample size
ax1.semilogx(n_samples, pi_estimates, 'bo-', markersize=8, label='Monte Carlo estimates')
ax1.axhline(y=np.pi, color='r', linestyle='--', linewidth=2, label='True value of π')
ax1.fill_between(n_samples, np.pi - 0.1, np.pi + 0.1, alpha=0.2, color='red', label='±0.1 error')
ax1.set_xlabel('Number of samples')
ax1.set_ylabel('Estimate of π')
ax1.set_title('Monte Carlo Estimation of π')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot error vs sample size
ax2.loglog(n_samples, errors, 'rs-', markersize=8, label='Actual error')
ax2.loglog(n_samples, [1/np.sqrt(n) for n in n_samples], 'k--', alpha=0.7, label='1/√n convergence')
ax2.set_xlabel('Number of samples')
ax2.set_ylabel('Absolute error')
ax2.set_title('Monte Carlo Error Convergence')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Summary

This notebook demonstrated three important numerical methods:

1. **Newton's Method**: Quadratic convergence for root finding when derivative is available
2. **Numerical Integration**: 
   - Trapezoidal rule: $O(h^2)$ convergence
   - Simpson's rule: $O(h^4)$ convergence
3. **Monte Carlo Integration**: $O(1/\sqrt{n})$ convergence, useful for high-dimensional problems

Each method has its own advantages and is suitable for different types of problems. The choice depends on the specific requirements of accuracy, computational cost, and problem characteristics.